# Crohn's extraction pipeline

Reads the 50 transcripts and runs each through the LLM extraction (schema-constrained,
resumable) -- first with Claude (the default, primary model), then with Gemini (the
comparison model) -- saving both sets of typed records + flat CSVs, and closing with
a side-by-side comparison frame. That's the whole scope of this notebook; analysis and
evaluation live elsewhere on purpose, so each notebook/script has one job:

- PharmaCorp's four answers + charts + ad-hoc queries: `python src/analysis.py` or `notebooks/03_analysis.ipynb`
- Evaluation (golden-set accuracy for both models, LLM-as-judge): `notebooks/02_evaluation.ipynb`

## 1 · Imports

In [ ]:
import os, json, sys
from pathlib import Path
import pandas as pd
# This notebook lives in notebooks/ but all paths (data/, src/) are repo-root-
# relative -- hop up one level if launched from inside notebooks/, no-op otherwise.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')
from schema import PatientRecord
import queries
print('imports ok · pandas', pd.__version__)


## 2 · Environment & API keys

In [ ]:
ENV_PATH = Path('.env').resolve()
loaded = False
try:
    from dotenv import load_dotenv
    loaded = load_dotenv(ENV_PATH)
except ImportError:
    print('!! python-dotenv not installed -> run:  pip install -r requirements.txt')
HAS_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
HAS_GEMINI = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
print(f'.env at {ENV_PATH}: {"found" if ENV_PATH.exists() else "MISSING"}  (loaded={loaded})')
print('ANTHROPIC_API_KEY detected:', HAS_KEY, ' |  GEMINI_API_KEY detected:', HAS_GEMINI)
if not HAS_KEY:
    print('\nNo Anthropic key found -- the Claude pass below will be cache-first only.')
if not HAS_GEMINI:
    print('No Gemini key found -- the Gemini pass below will be skipped (cache-first if data/cache_gemini/ already has records).')


## 3 · Model parameters

`provider/model` form. A bare `claude-...` id is auto-prefixed with `anthropic/` by
`extract.py`, but keeping the prefix here is the documented way. Each model gets its
own cache directory so the two runs never collide or skip each other's patients.

In [ ]:
CLAUDE_MODEL = 'anthropic/claude-sonnet-4-6'
GEMINI_MODEL = 'gemini/gemini-3.1-flash-lite'
TEMPERATURE = 0
MAX_TOKENS = 4000
CLAUDE_CACHE = Path('data/cache'); CLAUDE_CACHE.mkdir(parents=True, exist_ok=True)
GEMINI_CACHE = Path('data/cache_gemini'); GEMINI_CACHE.mkdir(parents=True, exist_ok=True)
print(dict(claude_model=CLAUDE_MODEL, gemini_model=GEMINI_MODEL, temperature=TEMPERATURE,
           max_tokens=MAX_TOKENS, claude_cache=str(CLAUDE_CACHE), gemini_cache=str(GEMINI_CACHE)))


## 4 · Load transcripts

In [ ]:
interviews = json.loads(Path('data/interviews.json').read_text())
print(len(interviews), 'transcripts')
ex = interviews[0]
print(ex['patient_id'], '—', ex['interview_transcript'][:160], '...')


## 5 · The extraction prompt (uncertainty instructions)

Imported from `src/extract.py` -- single source of truth, same prompt `extract_one`
uses for EITHER model. Provider-agnostic by design: schema-in-prompt + validate +
one self-correction retry, not a provider-native structured-output mode (see the
README's design decisions for why).

In [ ]:
from extract import SYSTEM_PROMPT
print(SYSTEM_PROMPT[:600], '...')


## 6 · The extraction function

`extract_one` (in `src/extract.py`) is prompt-based JSON: schema in the prompt,
validate against `PatientRecord`, self-correct once. Takes a `model=` argument, so
the exact same function drives both passes below.

In [ ]:
from extract import extract_one
print('extract_one imported from src/extract.py (prompt-based JSON + 1 retry, model-agnostic)')


## 7 · Run the pipeline — Claude (primary model)

Shares its loop with `src/run_extraction.py` (`run_extraction.run(...)`), so the
notebook and the terminal script can't drift apart: resumable via a per-patient
fingerprinted manifest, rate-limit retries that don't count as failures, stops
itself after genuine consecutive failures rather than grinding through the rest.
Safe to interrupt (Kernel -> Interrupt) at any point -- each success is written to
disk immediately.

In [ ]:
from run_extraction import run as run_extraction_pass

claude_result = run_extraction_pass(interviews, model=CLAUDE_MODEL, cache_dir=CLAUDE_CACHE)
if claude_result["failed"] or claude_result["stopped_early"]:
    print(f"\n{len(claude_result['failed'])} patient(s) still failing this pass "
          f"(stopped_early={claude_result['stopped_early']}): {claude_result['failed']}")
    print("Re-run this cell to retry just those -- everything else is skipped via the manifest.")
else:
    print("\nAll 50 patients extracted successfully with Claude.")


In [ ]:
# Load whatever is in the Claude cache right now -- works even if the run above is
# partial, so the rest of the notebook always has something to show.
claude_records = [PatientRecord.model_validate_json(f.read_text())
                   for f in sorted(CLAUDE_CACHE.glob('P*.json'))]
print(f'loaded {len(claude_records)}/{len(interviews)} Claude records from cache')


## 8 · Save state — Claude records + flat CSV frames

In [ ]:
OUT = Path('data/outputs'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'records.json').write_text(json.dumps([json.loads(r.model_dump_json()) for r in claude_records], indent=1))
patients, treatments, encounters = queries.to_frames(claude_records)
patients.to_csv(OUT / 'patients.csv', index=False)
treatments.to_csv(OUT / 'treatments.csv', index=False)
encounters.to_csv(OUT / 'encounters.csv', index=False)
print('wrote:', sorted(p.name for p in OUT.glob('*.csv')) + ['records.json'])
patients


## 9 · Run the pipeline — Gemini (comparison model)

Same resumable loop, same prompt, different model and its own cache directory so it
never touches the Claude cache. Skips cleanly if no Gemini key is set. Gemini's free-tier limits vary significantly by account/project (check your own
quota at aistudio.google.com rather than assuming a published default) --
`--throttle-seconds` isn't exposed as a notebook variable here, but if you hit
sustained rate limits, run the equivalent terminal command instead with
`--throttle-seconds <N>` tuned to your account, and re-open this notebook to
pick up the cache it wrote.

In [ ]:
if not HAS_GEMINI:
    print('no Gemini key -> skipping the Gemini pass (add GEMINI_API_KEY to .env to run it)')
    gemini_result = None
else:
    gemini_result = run_extraction_pass(interviews, model=GEMINI_MODEL, cache_dir=GEMINI_CACHE)
    if gemini_result["failed"] or gemini_result["stopped_early"]:
        print(f"\n{len(gemini_result['failed'])} patient(s) still failing this pass "
              f"(stopped_early={gemini_result['stopped_early']}): {gemini_result['failed']}")
        print("Re-run this cell to retry just those -- everything else is skipped via the manifest.")
    else:
        print("\nAll 50 patients extracted successfully with Gemini.")


In [ ]:
gemini_records = [PatientRecord.model_validate_json(f.read_text())
                   for f in sorted(GEMINI_CACHE.glob('P*.json'))]
print(f'loaded {len(gemini_records)}/{len(interviews)} Gemini records from cache')


## 10 · Save state — Gemini records + flat CSV frames

In [ ]:
if gemini_records:
    (OUT / 'records_gemini.json').write_text(
        json.dumps([json.loads(r.model_dump_json()) for r in gemini_records], indent=1))
    patients_gemini, treatments_gemini, encounters_gemini = queries.to_frames(gemini_records)
    patients_gemini.to_csv(OUT / 'patients_gemini.csv', index=False)
    treatments_gemini.to_csv(OUT / 'treatments_gemini.csv', index=False)
    encounters_gemini.to_csv(OUT / 'encounters_gemini.csv', index=False)
    print('wrote:', ['records_gemini.json', 'patients_gemini.csv', 'treatments_gemini.csv', 'encounters_gemini.csv'])
else:
    patients_gemini = pd.DataFrame()
    print('no Gemini records yet -- nothing to save')
patients_gemini


## 11 · Claude vs. Gemini — quick comparison frame

Side by side on the fields that drive PharmaCorp's four questions. Full scoring
(against the golden set, not just each other) lives in `02_evaluation.ipynb` --
this is a fast visual gut-check, not the real evaluation.

In [ ]:
if patients_gemini.empty:
    print('no Gemini records to compare against yet')
    cmp = None
else:
    cols = ['patient_id', 'on_biologic', 'biologic_status', 'referral_steps', 'completeness']
    cmp = patients[cols].merge(patients_gemini[cols], on='patient_id',
                                suffixes=('_claude', '_gemini'), how='outer')
    cmp['on_biologic_agrees'] = cmp['on_biologic_claude'] == cmp['on_biologic_gemini']
    print(f"on_biologic agreement: {cmp['on_biologic_agrees'].mean():.0%}  (n={len(cmp)})")
cmp


---

Extraction is done for both models: `data/outputs/records.json` / `records_gemini.json`
+ their flat CSVs are written. For PharmaCorp's four answers, charts, and ad-hoc
queries, run `python src/analysis.py` or step through `notebooks/03_analysis.ipynb`.
For golden-set accuracy (both models) and LLM-as-judge, see `notebooks/02_evaluation.ipynb`.